In [20]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_recall_curve, roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score, classification_report
)
import matplotlib.pyplot as plt

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


Device: cpu


In [21]:
# Feature sets exactly as defined in the design doc
VITALS = ['HR', 'O2Sat', 'Temp', 'MAP', 'Resp']
LABS = ['BUN', 'Chloride', 'Creatinine', 'Glucose', 'Hct', 'Hgb', 'WBC', 'Platelets']
STATICS = ['Age', 'Gender_1', 'ICULOS', 'HospAdmTime']
RAW_FEATURES = VITALS + LABS + STATICS
D_OBS = len(VITALS) + len(LABS)  # 13 masked features
D_FULL = len(RAW_FEATURES)       # 17 raw features

# Load datasets (Make sure these paths match your local setup)
df_a = pd.read_csv('data_part1.csv')
df_b = pd.read_csv('data_part2.csv')

# Ensure Gender is properly encoded to Gender_1
if 'Gender' in df_a.columns and 'Gender_1' not in df_a.columns:
    df_a['Gender_1'] = df_a['Gender']
    df_b['Gender_1'] = df_b['Gender']

print(f"Hospital A shape: {df_a.shape}")
print(f"Hospital B shape: {df_b.shape}")


Hospital A shape: (790215, 45)
Hospital B shape: (761995, 45)


In [22]:
# Hospital A splits
patient_ids_A = df_a['Patient_ID'].unique()

# 20% Test, 80% Train+Val
train_val_pids, test_pids = train_test_split(patient_ids_A, test_size=0.20, random_state=SEED)

# 90% Train, 10% Meta-Val from the Train+Val pool
train_pids, val_pids = train_test_split(train_val_pids, test_size=0.10, random_state=SEED)

print(f"Train patients: {len(train_pids)}")
print(f"Meta-Val patients: {len(val_pids)}")
print(f"Test patients A: {len(test_pids)}")
print(f"Test patients B: {df_b['Patient_ID'].nunique()}")


Train patients: 14641
Meta-Val patients: 1627
Test patients A: 4068
Test patients B: 20000


In [23]:
def build_patient_tensors(df_cohort, max_seq_len=168):
    """Builds (X, M, Delta, X_last, y) per patient as per the design doc."""
    seqs_X, seqs_M, seqs_Delta, seqs_X_last, seqs_y = [], [], [], [], []
    
    for pid, grp in df_cohort.groupby('Patient_ID'):
        grp = grp.sort_values('ICULOS')
        
        # Design Doc 2.1: Truncate keeping the most recent 168 hours
        if len(grp) > max_seq_len:
            grp = grp.tail(max_seq_len)
            
        T = len(grp)
        X_p = np.zeros((T, D_FULL), dtype=np.float32)
        M_p = np.zeros((T, D_OBS), dtype=np.float32)
        Delta_p = np.zeros((T, D_OBS), dtype=np.float32)
        X_last_p = np.zeros((T, D_OBS), dtype=np.float32)
        
        # Statics (always observed, never masked)
        for i, col in enumerate(STATICS):
            X_p[:, D_OBS + i] = grp[col].values
            
        # Masked features
        for d, col in enumerate(VITALS + LABS):
            vals = grp[col].values
            m = pd.notna(vals).astype(np.float32)
            M_p[:, d] = m
            X_p[:, d] = np.where(m == 1, vals, 0.0)
            
            last_obs = np.nan # Will become 0 after normalization
            for t in range(T):
                if t == 0:
                    Delta_p[t, d] = 0.0
                else:
                    if M_p[t-1, d] == 1:
                        Delta_p[t, d] = 1.0
                    else:
                        Delta_p[t, d] = Delta_p[t-1, d] + 1.0
                
                # X_last_t is the last observed value PRIOR to or at t
                X_last_p[t, d] = last_obs
                if m[t] == 1:
                    last_obs = vals[t]
                    
        y_p = grp['SepsisLabel'].values.astype(np.float32)
        
        seqs_X.append(X_p); seqs_M.append(M_p); seqs_Delta.append(Delta_p)
        seqs_X_last.append(X_last_p); seqs_y.append(y_p)
        
    return seqs_X, seqs_M, seqs_Delta, seqs_X_last, seqs_y


In [24]:
print("Building Train tensors...")
tr_X, tr_M, tr_Delta, tr_X_last, tr_y = build_patient_tensors(df_a[df_a['Patient_ID'].isin(train_pids)])

print("Building Val tensors...")
val_X, val_M, val_Delta, val_X_last, val_y = build_patient_tensors(df_a[df_a['Patient_ID'].isin(val_pids)])

print("Building Test A tensors...")
tea_X, tea_M, tea_Delta, tea_X_last, tea_y = build_patient_tensors(df_a[df_a['Patient_ID'].isin(test_pids)])

print("Building Test B tensors...")
teb_X, teb_M, teb_Delta, teb_X_last, teb_y = build_patient_tensors(df_b)

# Normalization based entirely on training fold (Section 4.3)
flat_tr_X = np.concatenate(tr_X, axis=0)
flat_tr_M = np.concatenate(tr_M, axis=0)

means = np.zeros(D_FULL, dtype=np.float32)
stds = np.ones(D_FULL, dtype=np.float32)

# Vitals/Labs (only use observed values for mean/std)
for d in range(D_OBS):
    obs_vals = flat_tr_X[flat_tr_M[:, d] == 1, d]
    if len(obs_vals) > 0:
        means[d] = obs_vals.mean()
        stds[d] = obs_vals.std() + 1e-6

# Statics
for i in range(D_OBS, D_FULL):
    means[i] = flat_tr_X[:, i].mean()
    stds[i] = flat_tr_X[:, i].std() + 1e-6

def apply_normalization(seqs_X, seqs_X_last, seqs_M):
    norm_X, norm_X_last = [], []
    for X_p, X_last_p, M_p in zip(seqs_X, seqs_X_last, seqs_M):
        X_norm = (X_p - means) / stds
        for d in range(D_OBS):
            X_norm[:, d] = np.where(M_p[:, d] == 1, X_norm[:, d], 0.0)
            
        # Catch any missing statics (Age, Gender, etc) and replace with mean (0.0)
        X_norm = np.nan_to_num(X_norm, nan=0.0)
            
        X_last_norm = (X_last_p - means[:D_OBS]) / stds[:D_OBS]
        X_last_norm = np.nan_to_num(X_last_norm, nan=0.0) 
        
        norm_X.append(X_norm); norm_X_last.append(X_last_norm)
    return norm_X, norm_X_last


tr_X, tr_X_last = apply_normalization(tr_X, tr_X_last, tr_M)
val_X, val_X_last = apply_normalization(val_X, val_X_last, val_M)
tea_X, tea_X_last = apply_normalization(tea_X, tea_X_last, tea_M)
teb_X, teb_X_last = apply_normalization(teb_X, teb_X_last, teb_M)
print("Normalization complete.")


Building Train tensors...
Building Val tensors...
Building Test A tensors...
Building Test B tensors...
Normalization complete.


In [25]:
class GRUDDataset(Dataset):
    def __init__(self, X, M, Delta, X_last, y):
        self.X, self.M, self.Delta, self.X_last, self.y = X, M, Delta, X_last, y
        self.lengths = [len(seq) for seq in X]
        
    def __len__(self): return len(self.X)
    def __getitem__(self, idx):
        return (torch.tensor(self.X[idx], dtype=torch.float32), torch.tensor(self.M[idx], dtype=torch.float32),
                torch.tensor(self.Delta[idx], dtype=torch.float32), torch.tensor(self.X_last[idx], dtype=torch.float32),
                torch.tensor(self.y[idx], dtype=torch.float32), self.lengths[idx])

def collate_fn(batch):
    Xs, Ms, Deltas, X_lasts, ys, lengths = zip(*batch)
    X_pad = torch.nn.utils.rnn.pad_sequence(Xs, batch_first=True, padding_value=0.0)
    M_pad = torch.nn.utils.rnn.pad_sequence(Ms, batch_first=True, padding_value=0.0)
    Delta_pad = torch.nn.utils.rnn.pad_sequence(Deltas, batch_first=True, padding_value=0.0)
    X_last_pad = torch.nn.utils.rnn.pad_sequence(X_lasts, batch_first=True, padding_value=0.0)
    y_pad = torch.nn.utils.rnn.pad_sequence(ys, batch_first=True, padding_value=0.0)
    return X_pad, M_pad, Delta_pad, X_last_pad, y_pad, torch.tensor(lengths, dtype=torch.long)

BATCH_SIZE = 64
train_loader = DataLoader(GRUDDataset(tr_X, tr_M, tr_Delta, tr_X_last, tr_y), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(GRUDDataset(val_X, val_M, val_Delta, val_X_last, val_y), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
tea_loader   = DataLoader(GRUDDataset(tea_X, tea_M, tea_Delta, tea_X_last, tea_y), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
teb_loader   = DataLoader(GRUDDataset(teb_X, teb_M, teb_Delta, teb_X_last, teb_y), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)


In [26]:
class GRUDCell(nn.Module):
    def __init__(self, input_size, hidden_size, obs_size):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.obs_size = obs_size

        # Decay params. W_gamma_x is element-wise (diagonal per paper §2.2).
        # Bias init small-positive so initial gamma is slightly < 1, not exactly 1.
        self.W_gamma_x = nn.Parameter(torch.zeros(obs_size))
        self.b_gamma_x = nn.Parameter(torch.full((obs_size,), 0.01))
        self.W_gamma_h = nn.Parameter(torch.zeros(hidden_size, obs_size))
        self.b_gamma_h = nn.Parameter(torch.full((hidden_size,), 0.01))

        self.W_z = nn.Linear(input_size, hidden_size)
        self.U_z = nn.Linear(hidden_size, hidden_size, bias=False)
        self.V_z = nn.Linear(obs_size, hidden_size, bias=False)

        self.W_r = nn.Linear(input_size, hidden_size)
        self.U_r = nn.Linear(hidden_size, hidden_size, bias=False)
        self.V_r = nn.Linear(obs_size, hidden_size, bias=False)

        self.W = nn.Linear(input_size, hidden_size)
        self.U = nn.Linear(hidden_size, hidden_size, bias=False)
        self.V = nn.Linear(obs_size, hidden_size, bias=False)

        # Smaller init on gate/cell linear maps reduces early-training logit magnitude.
        for lin in [self.W_z, self.W_r, self.W, self.U_z, self.U_r, self.U, self.V_z, self.V_r, self.V]:
            nn.init.xavier_uniform_(lin.weight, gain=0.5)
            if lin.bias is not None:
                nn.init.zeros_(lin.bias)

    def forward(self, x, m, delta, x_last, h_prev, rec_mask=None):
        # Clamp delta to a safe range. 168h cap is enforced upstream but defensive.
        delta = torch.clamp(delta, max=200.0)

        gamma_x = torch.exp(-torch.relu(self.W_gamma_x * delta + self.b_gamma_x))
        gamma_h = torch.exp(-torch.relu(F.linear(delta, self.W_gamma_h, self.b_gamma_h)))

        x_obs = x[:, :self.obs_size]
        x_imputed = m * x_obs + (1 - m) * (gamma_x * x_last)
        x_full = torch.cat([x_imputed, x[:, self.obs_size:]], dim=1)

        h_prev_decayed = gamma_h * h_prev
        if rec_mask is not None:
            h_prev_decayed = h_prev_decayed * rec_mask

        z = torch.sigmoid(self.W_z(x_full) + self.U_z(h_prev_decayed) + self.V_z(m))
        r = torch.sigmoid(self.W_r(x_full) + self.U_r(h_prev_decayed) + self.V_r(m))
        h_tilde = torch.tanh(self.W(x_full) + self.U(r * h_prev_decayed) + self.V(m))

        return (1 - z) * h_prev_decayed + z * h_tilde


class GRUDModel(nn.Module):
    def __init__(self, input_size=17, hidden_size=64, obs_size=13, drop=0.3):
        super().__init__()
        self.hidden_size = hidden_size
        self.cell = GRUDCell(input_size, hidden_size, obs_size)
        self.drop = drop
        self.classifier = nn.Sequential(nn.Dropout(0.5), nn.Linear(hidden_size, 1))
        # Small init on classifier so initial logits are O(1), not O(10).
        nn.init.xavier_uniform_(self.classifier[1].weight, gain=0.1)
        nn.init.zeros_(self.classifier[1].bias)

    def forward(self, X, M, Delta, X_last):
        B, T, _ = X.size()
        h = torch.zeros(B, self.hidden_size, device=X.device)

        # Variational recurrent dropout: one mask per sequence, shared across t.
        if self.training and self.drop > 0:
            rec_mask = (torch.rand(B, self.hidden_size, device=X.device) > self.drop).float() / (1 - self.drop)
        else:
            rec_mask = None

        outputs = []
        for t in range(T):
            h = self.cell(X[:, t, :], M[:, t, :], Delta[:, t, :], X_last[:, t, :], h, rec_mask)
            outputs.append(h.unsqueeze(1))

        outputs = torch.cat(outputs, dim=1)
        logits = self.classifier(outputs).squeeze(-1)
        # Clamp logits before BCE. With pos_weight=44, loss = pw * softplus(-logit) for pos
        # and softplus(logit) for neg — clamping at +/-15 keeps loss in a sane range.
        return torch.clamp(logits, min=-15.0, max=15.0)


model = GRUDModel(drop=0.1).to(device)
print(f'Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

Trainable params: 19,227


In [27]:
n_pos = sum((y == 1).sum() for y in tr_y)
n_neg = sum((y == 0).sum() for y in tr_y)
pw = torch.tensor([n_neg / (n_pos + 1e-6)], dtype=torch.float32).to(device)
print(f'pos_weight = {pw.item():.2f}')

criterion = nn.BCEWithLogitsLoss(pos_weight=pw)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

# Linear warmup over first 200 optimizer steps to absorb early instability.
WARMUP_STEPS = 200
global_step = 0

EPOCHS = 60
best_val_auprc = 0.0
patience_counter = 0

for ep in range(1, EPOCHS + 1):
    model.train()
    tl, valid_steps_train = 0.0, 0
    for Xb, Mb, Db, Xlb, yb, Lb in train_loader:
        Xb, Mb, Db, Xlb, yb, Lb = [v.to(device) for v in (Xb, Mb, Db, Xlb, yb, Lb)]

        # LR warmup
        if global_step < WARMUP_STEPS:
            lr_scale = (global_step + 1) / WARMUP_STEPS
            for pg in optimizer.param_groups:
                pg['lr'] = 1e-3 * lr_scale

        optimizer.zero_grad()
        logits = model(Xb, Mb, Db, Xlb)

        mask = (torch.arange(logits.size(1)).unsqueeze(0).to(device) < Lb.unsqueeze(1))
        loss = criterion(logits[mask], yb[mask])

        # NaN guard — skip the step if loss went sideways.
        if not torch.isfinite(loss):
            global_step += 1
            continue

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # tighter clip
        optimizer.step()
        global_step += 1

        tl += loss.item() * Lb.sum().item()
        valid_steps_train += Lb.sum().item()

    # Validation
    model.eval()
    vl, valid_steps_val = 0.0, 0
    val_probs, val_lbls = [], []
    with torch.no_grad():
        for Xb, Mb, Db, Xlb, yb, Lb in val_loader:
            Xb, Mb, Db, Xlb, yb, Lb = [v.to(device) for v in (Xb, Mb, Db, Xlb, yb, Lb)]
            logits = model(Xb, Mb, Db, Xlb)
            mask = (torch.arange(logits.size(1)).unsqueeze(0).to(device) < Lb.unsqueeze(1))
            loss = criterion(logits[mask], yb[mask])
            vl += loss.item() * Lb.sum().item()
            valid_steps_val += Lb.sum().item()
            probs = torch.sigmoid(logits).cpu().numpy()
            for i, L in enumerate(Lb.cpu().numpy()):
                val_probs.append(probs[i, :L])
                val_lbls.append(yb[i, :L].cpu().numpy())

    all_val_probs = np.concatenate(val_probs)
    all_val_lbls = np.concatenate(val_lbls)
    val_auprc = average_precision_score(all_val_lbls, all_val_probs)
    val_auroc = roc_auc_score(all_val_lbls, all_val_probs)

    scheduler.step(val_auprc)
    train_loss_avg = tl / max(valid_steps_train, 1)
    print(f'Epoch {ep:2d} | Train Loss: {train_loss_avg:.4f} | Val Loss: {vl/valid_steps_val:.4f} | Val AUPRC: {val_auprc:.4f} | Val AUROC: {val_auroc:.4f}')

    if val_auprc > best_val_auprc:
        best_val_auprc = val_auprc
        patience_counter = 0
        torch.save(model.state_dict(), 'grud_best.pt')
    else:
        patience_counter += 1
        if patience_counter >= 10:
            print("Early stopping triggered!")
            break

model.load_state_dict(torch.load('grud_best.pt'))
print("Best model loaded.")

pos_weight = 44.47
Epoch  1 | Train Loss: 1.2847 | Val Loss: 1.2649 | Val AUPRC: 0.0771 | Val AUROC: 0.7043
Epoch  2 | Train Loss: 1.1633 | Val Loss: 1.2245 | Val AUPRC: 0.0907 | Val AUROC: 0.7243
Epoch  3 | Train Loss: 1.1824 | Val Loss: 1.2121 | Val AUPRC: 0.0923 | Val AUROC: 0.7301
Epoch  4 | Train Loss: 1.1362 | Val Loss: 1.2076 | Val AUPRC: 0.0920 | Val AUROC: 0.7359
Epoch  5 | Train Loss: 1.2933 | Val Loss: 1.2150 | Val AUPRC: 0.0917 | Val AUROC: 0.7367
Epoch  6 | Train Loss: 1.1343 | Val Loss: 1.2034 | Val AUPRC: 0.0947 | Val AUROC: 0.7475
Epoch  7 | Train Loss: 1.2023 | Val Loss: 1.1876 | Val AUPRC: 0.0991 | Val AUROC: 0.7537
Epoch  8 | Train Loss: 1.1550 | Val Loss: 1.1906 | Val AUPRC: 0.1013 | Val AUROC: 0.7540
Epoch  9 | Train Loss: 1.2044 | Val Loss: 1.1794 | Val AUPRC: 0.1001 | Val AUROC: 0.7624
Epoch 10 | Train Loss: 1.2852 | Val Loss: 1.1792 | Val AUPRC: 0.1000 | Val AUROC: 0.7599
Epoch 11 | Train Loss: 1.2350 | Val Loss: 1.1831 | Val AUPRC: 0.1020 | Val AUROC: 0.7621
Ep

In [28]:
def find_optimal_threshold(y_true, y_prob, model_name="Model"):
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_prob)
    f1_scores = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-8)
    opt_th = thresholds[np.argmax(f1_scores)]

    clin_th = 0.50
    for i in range(len(recalls) - 1):
        if recalls[i] >= 0.60 and recalls[i+1] < 0.60:
            clin_th = thresholds[i]; break

    print(f"\n{'='*40}\n{model_name} — Threshold Analysis\n{'='*40}")
    for th_name, th in [("Optimal F1", opt_th), ("Clinical (>60% Recall)", clin_th)]:
        preds = (y_prob >= th).astype(int)
        print(f"{th_name} Threshold: {th:.4f}")
        print(f"  Precision: {precision_score(y_true, preds, zero_division=0):.4f}")
        print(f"  Recall:    {recall_score(y_true, preds):.4f}")
        print(f"  F1:        {f1_score(y_true, preds):.4f}\n")
    return opt_th, clin_th

def evaluate_loader(loader, name="Test"):
    model.eval()
    all_probs, all_lbls = [], []
    with torch.no_grad():
        for Xb, Mb, Db, Xlb, yb, Lb in loader:
            logits = model(*[v.to(device) for v in (Xb, Mb, Db, Xlb)])
            probs = torch.sigmoid(logits).cpu().numpy()
            for i, L in enumerate(Lb.numpy()):
                all_probs.append(probs[i, :L]); all_lbls.append(yb[i, :L].numpy())
                
    all_probs, all_lbls = np.concatenate(all_probs), np.concatenate(all_lbls)
    print(f"--- {name} Results ---")
    print(f"AUROC: {roc_auc_score(all_lbls, all_probs):.4f}")
    print(f"AUPRC: {average_precision_score(all_lbls, all_probs):.4f}")
    return find_optimal_threshold(all_lbls, all_probs, model_name=name)

print("Evaluating Hospital A Test Set...")
evaluate_loader(tea_loader, name="Hospital A Test")

print("\nEvaluating Hospital B Held-out Set...")
evaluate_loader(teb_loader, name="Hospital B Test")


Evaluating Hospital A Test Set...
--- Hospital A Test Results ---
AUROC: 0.7894
AUPRC: 0.0996

Hospital A Test — Threshold Analysis
Optimal F1 Threshold: 0.8327
  Precision: 0.1685
  Recall:    0.2529
  F1:        0.2023

Clinical (>60% Recall) Threshold: 0.5381
  Precision: 0.0648
  Recall:    0.6000
  F1:        0.1170


Evaluating Hospital B Held-out Set...
--- Hospital B Test Results ---
AUROC: 0.7739
AUPRC: 0.0745

Hospital B Test — Threshold Analysis
Optimal F1 Threshold: 0.8484
  Precision: 0.1135
  Recall:    0.2169
  F1:        0.1490

Clinical (>60% Recall) Threshold: 0.4596
  Precision: 0.0450
  Recall:    0.6000
  F1:        0.0838



(np.float32(0.8483859), np.float32(0.4596431))

In [29]:
# Save per-(Patient_ID, t_in_patient) probabilities so sepsis_ensemble can
# consume them without re-training GRU-D. Mirrors the (pid, t) keys the
# ensemble notebook expects.

def collect_probs_with_keys(loader, pid_order):
    """Returns a DataFrame with columns: Patient_ID, t_in_patient, prob, label."""
    model.eval()
    rows = []
    cursor = 0
    with torch.no_grad():
        for Xb, Mb, Db, Xlb, yb, Lb in loader:
            logits = model(*[v.to(device) for v in (Xb, Mb, Db, Xlb)])
            probs = torch.sigmoid(logits).cpu().numpy()
            for i, L in enumerate(Lb.numpy()):
                pid = pid_order[cursor]
                for t in range(L):
                    rows.append((pid, t, float(probs[i, t]), float(yb[i, t].item())))
                cursor += 1
    return pd.DataFrame(rows, columns=['Patient_ID', 't_in_patient', 'prob', 'label'])

# pandas groupby in build_patient_tensors yields patients in numpy-sorted order;
# replicate that here.
val_pid_order = sorted(df_a[df_a['Patient_ID'].isin(val_pids)]['Patient_ID'].unique())
tea_pid_order = sorted(df_a[df_a['Patient_ID'].isin(test_pids)]['Patient_ID'].unique())
teb_pid_order = sorted(df_b['Patient_ID'].unique())

collect_probs_with_keys(val_loader, val_pid_order).to_csv('grud_val_probs.csv', index=False)
collect_probs_with_keys(tea_loader, tea_pid_order).to_csv('grud_tea_probs.csv', index=False)
collect_probs_with_keys(teb_loader, teb_pid_order).to_csv('grud_teb_probs.csv', index=False)
print("Saved grud_val_probs.csv, grud_tea_probs.csv, grud_teb_probs.csv")

Saved grud_val_probs.csv, grud_tea_probs.csv, grud_teb_probs.csv
